<a href="https://colab.research.google.com/github/jordipech76-boop/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion03/sesion03_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 03 · Google Colab, Mágicos y Flujo de Entrega

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo sabe una máquina de compilación si un paso terminó bien o falló?
>
> **Objetivo de la sesión:** Escribir y leer archivos desde C y devolver códigos de salida POSIX, para que un pipeline de compilación pueda encadenar pasos y detenerse ante un error.

**Entregable:** tus funciones en artesano_sesion03_c.c, el harness en verde y la bitácora de la sesión

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Tu programa escribe un archivo de texto y luego cuenta sus líneas. El harness prueba los tres casos que rompen a casi todos: archivo sin salto final, archivo vacío y ruta inexistente.

Al final de la sesión tendrás: **Tres funciones en C que escriben y leen archivos y que reportan el código de salida real de un comando externo.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 03](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion03.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion03.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion03.h"

int escribir_texto(const char *ruta, const char *texto);
int contar_lineas(const char *ruta);
```

- Usa fopen/fgetc o fgets y cierra siempre el archivo.
- Un archivo inexistente se reporta con -1, no con un aborto.
- Una última línea sin salto de línea también cuenta.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion03_c.c` junto con `artesano_sesion03_c.c`
> (o `ia_sesion03_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `escribir_texto(...)==0` | 0 |
| 2 | `contar_lineas(3 lineas)` | 3 |
| 3 | `contar_lineas(sin salto final)` | 2 |
| 4 | `contar_lineas(archivo vacio)` | 0 |
| 5 | `contar_lineas(inexistente)` | -1 |
| 6 | `escribir_texto(ruta invalida)` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 03 - Archivos: escribir y contar lineas ===
Caso                               esperado:              obtenido:              resultado
[ 1] escribir_texto(...)==0             esperado: 0            obtenido: 0            OK
[ 2] contar_lineas(3 lineas)            esperado: 3            obtenido: 3            OK
[ 3] contar_lineas(sin salto final)     esperado: 2            obtenido: 2            OK
[ 4] contar_lineas(archivo vacio)       esperado: 0            obtenido: 0            OK
[ 5] contar_lineas(inexistente)         esperado: -1           obtenido: -1           OK
[ 6] escribir_texto(ruta invalida)      esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Los archivos se cierran en todos los caminos y un error se reporta con -1.
- [ ] contar_lineas distingue un archivo vacío, uno con salto final y uno sin él.
- [ ] escribir_texto falla con -1 si la ruta no se puede abrir.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion03_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion03.h
/* api_sesion03.h - contrato de la Sesion 03
 * Archivos: escribir y contar líneas
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION03_H
#define API_SESION03_H

int escribir_texto(const char *ruta, const char *texto);
int contar_lineas(const char *ruta);

#endif /* API_SESION03_H */


Writing api_sesion03.h


In [4]:
%%writefile artesano_sesion03_c.c
/* artesano_sesion03_c.c - Fase 1: El Artesano (sin IA)
 * Archivos: escribir y contar líneas.
 *
 * Implementa lo que declara api_sesion03.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion03.h"
#include <stdio.h>
#include <string.h>

int escribir_texto(const char *ruta, const char *texto) {
    FILE *f = fopen(ruta, "w");
    if (f == NULL) {
        return -1;              /* no se pudo abrir (ruta invalida) */
    }
    if (fputs(texto, f) == EOF) {
        fclose(f);
        return -1;
    }
    if (fclose(f) != 0) {
        return -1;
    }
    return 0;
}

int contar_lineas(const char *ruta) {
    FILE *f = fopen(ruta, "r");
    if (f == NULL) {
        return -1;              /* archivo inexistente */
    }
    int lineas = 0;
    int c;
    int anterior = '\n';
    while ((c = fgetc(f)) != EOF) {
        if (c == '\n') {
            lineas++;
        }
        anterior = c;
    }
    if (anterior != '\n') {
        lineas++;
    }
    fclose(f);
    return lineas;
}

Overwriting artesano_sesion03_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion03_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [5]:
%%writefile ia_sesion03_c.c
/* ia_sesion03_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion03.h. No puede traer main().
 */
#include "api_sesion03.h"

/* TODO: pega aqui la propuesta de la IA */


Writing ia_sesion03_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [6]:
%%writefile harness_sesion03_c.c
/* harness_sesion03_c.c - auditoria automatica de la Sesion 03
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion03_c.c artesano_sesion03_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion03.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 03 - Archivos: escribir y contar lineas ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = escribir_texto("datos.txt", "uno\ndos\ntres\n"); verificar_int("escribir_texto(...)==0", (long)(0), (long)(r)); }
    { escribir_texto("datos.txt", "uno\ndos\ntres\n"); verificar_int("contar_lineas(3 lineas)", (long)(3), (long)(contar_lineas("datos.txt"))); }
    { escribir_texto("datos2.txt", "a\nb"); verificar_int("contar_lineas(sin salto final)", (long)(2), (long)(contar_lineas("datos2.txt"))); }
    { escribir_texto("vacio.txt", ""); verificar_int("contar_lineas(archivo vacio)", (long)(0), (long)(contar_lineas("vacio.txt"))); }
    {  verificar_int("contar_lineas(inexistente)", (long)(-1), (long)(contar_lineas("no_existe.txt"))); }
    {  verificar_int("escribir_texto(ruta invalida)", (long)(-1), (long)(escribir_texto("/no/existe/x.txt", "a"))); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md" "\n");
    return 0;
}


Writing harness_sesion03_c.c


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion03_c.c artesano_sesion03_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 03 - Archivos: escribir y contar lineas ===
Caso                               esperado:              obtenido:              resultado
[ 1] escribir_texto(...)==0             esperado: 0            obtenido: 0            OK
[ 2] contar_lineas(3 lineas)            esperado: 3            obtenido: 3            OK
[ 3] contar_lineas(sin salto final)     esperado: 2            obtenido: 2            OK
[ 4] contar_lineas(archivo vacio)       esperado: 0            obtenido: 0            OK
[ 5] contar_lineas(inexistente)         esperado: -1           obtenido: -1           OK
[ 6] escribir_texto(ruta invalida)      esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md


In [8]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion03_c.c ia_sesion03_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 03 - Archivos: escribir y contar lineas ===
Caso                               esperado:              obtenido:              resultado
[ 1] escribir_texto(...)==0             esperado: 0            obtenido: 0            OK
[ 2] contar_lineas(3 lineas)            esperado: 3            obtenido: 3            OK
[ 3] contar_lineas(sin salto final)     esperado: 2            obtenido: 2            OK
[ 4] contar_lineas(archivo vacio)       esperado: 0            obtenido: 0            OK
[ 5] contar_lineas(inexistente)         esperado: -1           obtenido: -1           OK
[ 6] escribir_texto(ruta invalida)      esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion03.md` a `auditorias/auditoria_sesion03.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
